# Quick one-stage annulus ablation

This is a short screening experiment for the annulus bottleneck. It compares the current one-stage setup with annulus-balanced sampling and annulus boundary weighting at slight training noise (`sigma=0.001`). Noise is applied only to training gradients; validation and test gradients remain clean. Use this notebook to choose a candidate configuration before running the full final model.

In [1]:
from pathlib import Path
import gc
import json
import sys
import pandas as pd
import torch

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from config import OneModelRunConfig, Stage2ModelConfig, StageTrainingConfig
from final_models.one_stage import run_one_stage

N = 10
SIGMA = 0.001
SEED = 42
TRAINING_SAMPLES = 5_000
VALIDATION_SAMPLES = 500
TEST_SAMPLES = 500
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUTPUT_ROOT = ROOT / 'outputs' / 'one_stage_annulus_ablation_quick'
NOTES_ROOT = ROOT / 'docs' / 'experiments' / 'sigma01_results'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
NOTES_ROOT.mkdir(parents=True, exist_ok=True)
torch.set_num_threads(2)
print('Device:', DEVICE)
print('Training noise only; validation/test remain clean')

Device: cuda
Training noise only; validation/test remain clean


In [2]:
def make_config(name, shape_weights, annulus_edge_weight):
    model = Stage2ModelConfig(
        hidden_layer_sizes=(512, 1024), dropout_rates=(0.1, 0.1),
        model_type='coord_conv_decoder', latent_grid_size=16, latent_channels=160,
        decoder_channels=(160, 128, 96, 64, 32),
        use_rectangle_edge_weighting=True, use_foreground_pos_weight=False,
        rectangle_edge_weight=4.0, rectangle_edge_width=3, edge_weight_mode='all',
        annulus_edge_weight=annulus_edge_weight, annulus_edge_width=3,
        training=StageTrainingConfig(
            epochs=80, batch_size=96, learning_rate=0.0005, validation_frequency=20,
            verbose=False, early_stopping_patience=15, min_epochs=30,
            min_improvement=0.001, lr_drop_factor=0.5, lr_drop_period=40,
            weight_decay=0.00025, gradient_clip_norm=0.8, loss_type='bce_dice',
            dice_loss_weight=1.0, dice_smooth=1.0,
        ),
    )
    return OneModelRunConfig(
        N=N, training_samples=TRAINING_SAMPLES, validation_samples=VALIDATION_SAMPLES,
        test_samples=TEST_SAMPLES, noise_sigma=SIGMA, noise_mode='absolute', seed=SEED,
        training_noise_replicas=1, training_shape_weights=shape_weights,
        use_validation_threshold_sweep=True, model=model,
        output_dir=OUTPUT_ROOT / name,
    )

BASELINE_WEIGHTS = (
    ('rectangle', 0.25), ('two_circles', 0.45), ('annulus', 0.10),
    ('ellipse', 0.15), ('circle', 0.05),
)
ANNULUS_BALANCED_WEIGHTS = (
    ('rectangle', 0.25), ('two_circles', 0.40), ('annulus', 0.20),
    ('ellipse', 0.10), ('circle', 0.05),
)

conditions = {
    'baseline': (BASELINE_WEIGHTS, 1.0),
    'annulus_balanced': (ANNULUS_BALANCED_WEIGHTS, 1.0),
    'annulus_balanced_weighted': (ANNULUS_BALANCED_WEIGHTS, 2.0),
}
print('Conditions:', ', '.join(conditions))

Conditions: baseline, annulus_balanced, annulus_balanced_weighted


In [3]:
results = []
shape_results = []
summaries = {}

for name, (shape_weights, annulus_edge_weight) in conditions.items():
    print(f'\n=== QUICK ONE-STAGE: {name} ===', flush=True)
    config = make_config(name, shape_weights, annulus_edge_weight)
    summary = run_one_stage(config, device=DEVICE)
    summaries[name] = summary
    row = {
        'condition': name,
        'sigma': SIGMA,
        'training_samples': TRAINING_SAMPLES,
        'test_iou': summary['metrics']['test']['mean_iou'],
        'fixed_iou': summary['metrics']['fixed']['mean_iou'],
        'validation_iou': summary['threshold_summary']['validation_metrics']['mean_iou'],
        'threshold': summary['threshold_summary']['selected_threshold'],
    }
    results.append(row)
    for shape, metrics in summary['metrics_by_shape']['test'].items():
        shape_results.append({'condition': name, 'shape': shape, **metrics})
    print(json.dumps(row, indent=2), flush=True)
    print(pd.DataFrame([r for r in shape_results if r['condition'] == name]).to_string(index=False), flush=True)
    del summary, config
    gc.collect()

comparison = pd.DataFrame(results).sort_values('test_iou', ascending=False)
by_shape = pd.DataFrame(shape_results).sort_values(['condition', 'shape'])
comparison.to_csv(NOTES_ROOT / 'one_stage_annulus_ablation_quick.csv', index=False)
by_shape.to_csv(NOTES_ROOT / 'one_stage_annulus_ablation_quick_by_shape.csv', index=False)
(NOTES_ROOT / 'one_stage_annulus_ablation_quick.json').write_text(json.dumps(results, indent=2), encoding='utf-8')
print('\n=== QUICK COMPARISON ===')
print(comparison.to_string(index=False))
print('\n=== QUICK PER-SHAPE RESULTS ===')
print(by_shape.to_string(index=False))
comparison


=== QUICK ONE-STAGE: baseline ===
{
  "condition": "baseline",
  "sigma": 0.001,
  "training_samples": 5000,
  "test_iou": 0.8054290279683926,
  "fixed_iou": 0.8677965135741923,
  "validation_iou": 0.8100371542817013,
  "threshold": 0.55
}
condition       shape  sample_count  mean_iou  pixel_accuracy
 baseline     annulus            66  0.635908        0.945845
 baseline      circle            29  0.952731        0.990739
 baseline     ellipse            63  0.873100        0.985832
 baseline   rectangle           120  0.792148        0.985937
 baseline two_circles           222  0.824560        0.987423

=== QUICK ONE-STAGE: annulus_balanced ===
{
  "condition": "annulus_balanced",
  "sigma": 0.001,
  "training_samples": 5000,
  "test_iou": 0.796514633572516,
  "fixed_iou": 0.8992428185971081,
  "validation_iou": 0.8047410944859922,
  "threshold": 0.55
}
       condition       shape  sample_count  mean_iou  pixel_accuracy
annulus_balanced     annulus            62  0.677828        0.

,condition,sigma,training_samples,test_iou,fixed_iou,validation_iou,threshold
0,baseline,0.001,5000,0.805429,0.867797,0.810037,0.55
1,annulus_balanced,0.001,5000,0.796515,0.899243,0.804741,0.55
2,annulus_balanced_weighted,0.001,5000,0.793428,0.900019,0.802324,0.50


## Promotion rule

Promote a condition to the full final run only if it improves annulus test IoU without materially reducing overall random-test IoU. Treat fixed IoU as secondary. The quick run is directional because it uses fewer samples and epochs; repeat the selected configuration with the full final settings before reporting it.